In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install mne pyedflib numpy pandas matplotlib scikit-learn tensorflow

In [ ]:
import os
import numpy as np
import mne
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, 
                                     BatchNormalization, LSTM, Bidirectional,
                                     Dense, Dropout, Attention, GlobalAveragePooling1D)
from tensorflow.keras.utils import to_categorical

In [ ]:
def load_edf(file_path):
    raw = mne.io.read_raw_edf(file_path, preload=True, verbose=False)
    data = raw.get_data()
    return data

In [ ]:
def preprocess_signal(signal, sfreq=256):
    # Bandpass filter (0.5–40 Hz)
    filtered = mne.filter.filter_data(signal, sfreq, 0.5, 40)
    
    # Normalize
    mean = np.mean(filtered, axis=1, keepdims=True)
    std = np.std(filtered, axis=1, keepdims=True)
    normalized = (filtered - mean) / std
    
    return normalized

In [ ]:
def segment_signal(signal, window_size=256*5):  # 5 sec window
    segments = []
    
    for i in range(0, signal.shape[1] - window_size, window_size):
        segment = signal[:, i:i+window_size]
        segments.append(segment)
    
    return np.array(segments)

In [ ]:
def create_labels(num_segments):
    # 0 = non-seizure, 1 = seizure
    labels = np.zeros(num_segments)
    labels[:num_segments//5] = 1   # assume some seizure data
    return labels

In [ ]:
X = []
y = []

data_path = "/kaggle/input/datasets/datasetengineer/epilepsy-dataset/epilepsy_federated_dataset.csv"  # update path

for file in os.listdir(data_path):
    if file.endswith(".edf"):
        raw = load_edf(os.path.join(data_path, file))
        processed = preprocess_signal(raw)
        segments = segment_signal(processed)
        labels = create_labels(len(segments))
        
        X.extend(segments)
        y.extend(labels)

X = np.array(X)
y = np.array(y)

In [ ]:
X = X.transpose(0, 2, 1)  # (samples, time, channels)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y)

y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [ ]:
input_layer = Input(shape=(X_train.shape[1], X_train.shape[2]))

# CNN
x = Conv1D(64, 3, activation='relu')(input_layer)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)

x = Conv1D(128, 3, activation='relu')(x)
x = BatchNormalization()(x)
x = MaxPooling1D(2)(x)

# BiLSTM
x = Bidirectional(LSTM(64, return_sequences=True))(x)

# Attention
attention = Attention()([x, x])

# Pooling
x = GlobalAveragePooling1D()(attention)

# Dense
x = Dense(64, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(2, activation='softmax')(x)

model = Model(inputs=input_layer, outputs=output)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32
)

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true, y_pred)
print(cm)

print(classification_report(y_true, y_pred))